# Will They Buy an EV? Boosting Algorithms from Scratch

Binary classification: predict whether a person will buy an electric vehicle (`Will_Buy_EV`) from demographic, financial, and charging-infrastructure features.

The goal is to **build AdaBoost from scratch** (NumPy + pandas) and compare it against scikit-learn's `AdaBoostClassifier`, using **ROC AUC** as the metric.

**Workflow**
1. Load and inspect the data
2. Preprocess (ordinal, binary, one-hot encoding, scaling)
3. Train `AdaBoostScratch` (decision stumps as weak learners)
4. Evaluate with ROC AUC and compare with scikit-learn
5. Score the Kaggle-style `test.csv`

**Files:** `boosting.py` (`AdaBoostScratch`) and `decisions_stump.py` (`DecisionStump`) sit next to this notebook.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from boosting import AdaBoostScratch

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import roc_auc_score

## 2. Load the data

`train.csv` has the target `Will_Buy_EV`; `test.csv` does not. Set `DATA_DIR` to wherever the CSVs live on your machine.

In [3]:
DATA_DIR = Path(r"C:\Users\DELL\Datasets\Will_buy_EV_dataset")

df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

## 3. Exploratory data analysis

Quick look at both files, dtypes, missing values, and summary statistics.

In [4]:
df.head(10)

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
5,5,50,95938.0,55.1,2,8,15,3.0,Other,Urban,Sedan,No,No,Low,No
6,6,55,109219.0,42.9,1,1,5,4.0,Male,Suburban,Sedan,Yes,Yes,Low,Yes
7,7,45,60796.0,52.2,2,13,3,2.0,Other,Urban,SUV,No,Yes,Medium,No
8,8,39,67725.0,31.5,2,11,3,2.0,Female,Urban,Sedan,No,Yes,Medium,No
9,9,57,114827.0,66.9,2,0,1,5.0,Male,Rural,Sedan,Yes,Yes,Low,Yes


In [5]:
test_df.head(10)

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
0,668665,61,67725.0,16.9,2,7,4,4.0,Male,Suburban,Sedan,Yes,No,Low
1,668666,42,152835.0,41.9,2,9,9,4.0,Male,Urban,SUV,No,No,Low
2,668667,68,86877.0,53.3,1,10,11,4.0,Female,Urban,Sedan,No,No,Low
3,668668,39,46794.0,34.1,2,4,8,4.0,Female,Suburban,Sedan,Yes,No,Low
4,668669,55,112172.0,57.2,1,6,4,1.0,Female,Suburban,Sedan,Yes,Yes,Low
5,668670,61,94863.0,5.0,2,1,2,3.0,Male,Rural,Sedan,Yes,No,Low
6,668671,29,89609.0,47.1,2,5,7,4.0,Male,Suburban,Sedan,Yes,Yes,Low
7,668672,62,95591.0,32.8,2,11,9,2.0,Male,Urban,SUV,No,No,Medium
8,668673,39,73462.0,5.0,2,1,3,5.0,Female,Suburban,Sedan,Yes,Yes,Low
9,668674,60,57534.0,36.3,2,0,6,1.0,Male,Suburban,SUV,Yes,Yes,Low


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  object 
 9   City_Type                    668665 non-null  object 
 10  Current_Car_Type             668665 non-null  object 
 11  Home_Charging_Possible       668665 non-null  object 
 12  Subsidy_Available            668665 non-null  object 
 13 

**Observations**
- ~669k training rows, 15 columns, no missing values (confirmed below).
- 7 numeric columns, 6 categorical/yes-no columns, plus the target (`Yes`/`No`) and an `id` column.
- `id` is only a row index, so it carries no signal and is dropped before modelling.

In [7]:
df.isna().sum()

id                             0
Age                            0
Annual_Income_USD              0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Environmental_Concern_Level    0
Gender                         0
City_Type                      0
Current_Car_Type               0
Home_Charging_Possible         0
Subsidy_Available              0
Range_Anxiety_Level            0
Will_Buy_EV                    0
dtype: int64

In [8]:
df.columns

Index(['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km',
       'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
       'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender',
       'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level', 'Will_Buy_EV'],
      dtype='object')

In [9]:
df.describe()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level
count,668665.00000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000
mean,334332.00000,47.039171,84769.266989,32.158298,1.712626,4.960408,7.176314,2.935477
std,193027.10321,12.875448,28648.029042,18.730474,0.729275,3.926843,5.186627,1.429119
min,0.00000,25.000000,30000.000000,5.000000,1.000000,0.000000,0.000000,1.000000
25%,167166.00000,36.000000,67376.000000,17.200000,1.000000,2.000000,3.000000,2.000000
50%,334332.00000,47.000000,84880.000000,33.600000,2.000000,4.000000,6.000000,3.000000
75%,501498.00000,58.000000,102753.000000,47.400000,2.000000,7.000000,10.000000,4.000000
max,668664.00000,69.000000,188549.000000,98.700000,4.000000,14.000000,19.000000,5.000000


## 4. Feature groups

Checking the unique values of each categorical column decides how each one is encoded:

| Group | Columns | Encoding | Why |
|---|---|---|---|
| `CAT` | `Gender`, `City_Type`, `Current_Car_Type` | One-hot | No natural order |
| `CAT_2` | `Range_Anxiety_Level` | Ordinal (`Low`=0, `Medium`=1, `High`=2) | Real order |
| `BOOL` | `Home_Charging_Possible`, `Subsidy_Available` | `No`=0, `Yes`=1 | Binary |
| `NUM` | 7 numeric columns | Standardised | Needed for scale-sensitive models; harmless for trees |

In [10]:
COLS = ['Gender','City_Type', 'Current_Car_Type', 'Home_Charging_Possible','Subsidy_Available', 'Range_Anxiety_Level', 'Will_Buy_EV']

In [11]:
for cols in COLS:
    print(df[cols].unique())

['Male' 'Female' 'Other']
['Suburban' 'Rural' 'Urban']
['Sedan' 'SUV' 'Hatchback' 'Truck']
['Yes' 'No']
['No' 'Yes']
['Low' 'Medium' 'High']
['No' 'Yes']


In [12]:
TARGET = 'Will_Buy_EV'
CAT = ['Gender','City_Type', 'Current_Car_Type']
CAT_2 = 'Range_Anxiety_Level'
BOOL = ['Home_Charging_Possible','Subsidy_Available']
NUM = ['Age', 'Annual_Income_USD', 'Daily_Commute_km',
       'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
       'Charging_Stations_Near_Work', 'Environmental_Concern_Level']

## 5. Preprocessing

Two functions:

- `encode(d)`: deterministic transforms only (ordinal map, yes/no map, one-hot). Learns nothing from the data, so it is safe to apply to train, validation, and test alike.
- `preprocess(df)`: stratified 80/20 split, `encode` on each part, then the **scaler is fitted on train only** and reused (`transform`) on validation. The validation columns are reindexed to match train so a category missing from one split can't shift the feature layout.

The `id` column is dropped, and `assert` catches any unmapped value (it would become `NaN`).

In [13]:
def encode(d):
    d = d.copy()
    d[CAT_2] = d[CAT_2].map({'Low': 0, 'Medium': 1, 'High': 2})
    for bol in BOOL:
        d[bol] = d[bol].map({'No': 0, 'Yes': 1})
    return pd.get_dummies(d, columns=CAT, dtype=int)


def preprocess(df):
    df = df.drop(columns=['id']).copy()
    if not pd.api.types.is_numeric_dtype(df[TARGET]):    # 'Yes'/'No' strings -> 1/0
        df[TARGET] = df[TARGET].map({'No': 0, 'Yes': 1})

    train, valid = train_test_split(df, test_size=0.2, random_state=42, stratify=df[TARGET])
    train, valid = encode(train), encode(valid)
    valid = valid.reindex(columns=train.columns, fill_value=0)   # align one-hot columns

    sc = StandardScaler()
    train[NUM] = sc.fit_transform(train[NUM])    # fit on train only
    valid[NUM] = sc.transform(valid[NUM])        # reuse train statistics

    assert not train.isna().any().any() and not valid.isna().any().any(), "unmapped value -> NaN"

    x_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
    x_valid, y_valid = valid.drop(columns=[TARGET]), valid[TARGET]
    return x_train, y_train, x_valid, y_valid, sc

In [14]:
x_train, y_train, x_valid, y_valid, sc = preprocess(df)
print(x_train.shape, x_valid.shape)
print(y_train.mean())   # share of buyers, should match in both splits

(534932, 20) (133733, 20)
0.17464462772838416


## 6. Label encoding for `AdaBoostScratch`

`AdaBoostScratch` and its `DecisionStump` expect labels in **{-1, +1}**: the sample re-weighting `exp(-alpha * y * pred)` and the stump's error masks (`y == 1`, `y == -1`) only work with that encoding. With {0, 1} labels, class 0 gets zero weight everywhere and the model learns nothing, which gives an AUC of exactly 0.5.

So the scratch model is trained on `y_train_pm`. Evaluation and scikit-learn keep the original {0, 1} labels.

`x_small` is an optional 25% stratified sample for quick sanity checks before a full fit.

In [15]:
y_train_pm = np.where(y_train == 1, 1, -1)
print(np.unique(y_train_pm))   # must be [-1  1]

x_small, _, y_small_pm, _ = train_test_split(
    x_train, y_train_pm, train_size=0.25, stratify=y_train_pm, random_state=42)

[-1  1]


## 7. Evaluation helper

ROC AUC needs **continuous scores**. Hard `-1/+1` predictions collapse the ROC curve to one point and give ~0.5 regardless of quality. `get_scores` prefers `predict_proba`, then `decision_function`, and only falls back to `predict` (with a warning).

In [16]:
def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        s = model.predict_proba(X)
        return s[:, 1] if s.ndim == 2 else s
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    print("WARNING: using hard labels, AUC will be misleading")
    return model.predict(X)


def get_roc(model, X, y):
    auc = roc_auc_score(y, get_scores(model, X))
    print("ROC_AUC_SCORE :", auc)
    return auc

## 8. AdaBoost from scratch

First a quick 10-stump run on the 25% sample to check everything works, then the full fit.

In [17]:
quick = AdaBoostScratch(n_estimators=10).fit(x_small, y_small_pm)
get_roc(quick, x_valid, y_valid)

ROC_AUC_SCORE : 0.775672919747685


np.float64(0.775672919747685)

In [18]:
%%time
ada = AdaBoostScratch()                     # default n_estimators=50
model = ada.fit(x_train, y_train_pm)        # {-1, +1} labels

CPU times: total: 52.6 s
Wall time: 52.9 s


In [19]:
get_roc(model, x_valid, y_valid)

ROC_AUC_SCORE : 0.775672919747685


np.float64(0.775672919747685)

## 9. Baseline: scikit-learn `AdaBoostClassifier`

Same data, same split, default settings, original {0, 1} labels.

In [20]:
ada_2 = AdaBoostClassifier()
model_2 = ada_2.fit(x_train, y_train)
get_roc(model_2, x_valid, y_valid)

ROC_AUC_SCORE : 0.9358174405419402


np.float64(0.9358174405419402)

## 10. Results

| Model | Validation ROC AUC |
|---|---|
| scikit-learn `AdaBoostClassifier` | 0.9358 |
| `AdaBoostScratch` | *0.7757* |

The scratch model should land close to scikit-learn. A small gap is expected: both use stumps, but the split search and the scikit-learn default algorithm differ slightly. A value of exactly 0.5 means the labels or scores are wrong (see section 6 and 7).

## 11. Score the test set

`test.csv` has no target. It goes through the same `encode` + the **train-fitted** scaler, and its columns are aligned to the training columns.

In [21]:
test_ids = test_df['id']
x_test = encode(test_df.drop(columns=['id'])).reindex(columns=x_train.columns, fill_value=0)
x_test[NUM] = sc.transform(x_test[NUM])

submission = pd.DataFrame({'id': test_ids, TARGET: get_scores(model, x_test)})
submission.head()

,id,Will_Buy_EV
0,668665,0.301612
1,668666,0.923575
2,668667,0.301612
3,668668,0.301612
4,668669,0.053240
